# SHAPE — Notebook 3 of 4 · Tools & Environment

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

**Coverage (rubric):** Two external tools · Input processing · Error handling · Safety & boundaries  
**Imports from:** `shape.tools`, `shape.safety`, `shape.sensors`


In [ ]:
import os, sys, subprocess
from pathlib import Path
if not Path('shape').exists() and Path('../shape').exists():
    os.chdir('..')
subprocess.run([sys.executable, '-m', 'pip', 'install',
                '-r', 'requirements.txt', '--quiet'], check=True)
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.makedirs('workspace', exist_ok=True)
print('Root:', root)

In [ ]:
# ── Ollama setup (Colab only) ─────────────────────────────────────────────────
# Installs Ollama inside the Colab VM, pulls CodeLlama 7B, and starts the server.
# Skip this cell if running locally with Ollama already running.
# First run takes ~5 minutes (model download). Subsequent runs: ~30 seconds.

import subprocess, time, urllib.request, urllib.error

def _ollama_running():
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        return True
    except Exception:
        return False

try:
    import google.colab  # only runs in Colab
    if not _ollama_running():
        print('Installing Ollama...')
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                       shell=True, check=True, capture_output=True)
        print('Starting Ollama server...')
        subprocess.Popen(['ollama', 'serve'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if _ollama_running(): break
            time.sleep(1)
        print('Pulling codellama (this may take a few minutes)...')
        subprocess.run(['ollama', 'pull', 'codellama'], check=True)
        print('Ollama ready. ✓')
    else:
        print('Ollama already running. ✓')
except ImportError:
    if _ollama_running():
        print('Local Ollama detected. ✓')
    else:
        print('WARNING: Ollama not running. Start it with: ollama serve')
        print('Then run: ollama pull codellama')

## Tool 1: `code_executor` — Sandboxed Python Execution

| Property | Value |
|---|---|
| **Trigger** | LLM emits `Action: code_executor` |
| **Execution** | `subprocess.run([sys.executable, tmp_file], timeout=10)` |
| **Output** | `stdout + stderr`, capped at 4,096 chars |
| **Empty input** | Returns `'[CodeExecutor] Empty code — nothing to run.'` |
| **Timeout** | Returns `'[CodeExecutor] Execution timed out after N s.'` |
| **Crash** | Returns `'[CodeExecutor] Error: <exc>'` |
| **Output interpretation** | Agent reads stdout as confirmation; stderr as error evidence for next step |


In [ ]:
from shape.tools.code_executor import CodeExecutorTool
import time

executor = CodeExecutorTool({'executor_timeout': 10, 'executor_max_output': 4096})
timeout_ex = CodeExecutorTool({'executor_timeout': 2,  'executor_max_output': 512})

cases = [
    ('Normal execution',         executor,   'print(2 ** 10)'),
    ('Empty input',              executor,   ''),
    ('Timeout (2 s budget)',     timeout_ex, 'import time; time.sleep(999)'),
    ('Syntax error → stderr',    executor,   'def bad(:\n    pass'),
    ('Runtime error → stderr',   executor,   'x = 1/0'),
    ('Output cap (5000 chars)',   executor,   'print("A" * 5000)'),
]

for label, tool, code in cases:
    t0  = time.perf_counter()
    out = tool._run(code)
    ms  = (time.perf_counter() - t0) * 1000
    print(f'[{label}]  ({ms:.0f} ms)')
    print(f'  → {repr(out[:90])}')
    print()

## Tool 2: `doc_search` — Offline Semantic Search

| Property | Value |
|---|---|
| **Trigger** | LLM emits `Action: doc_search` |
| **Primary** | FAISS `similarity_search(query, k=3)` on `all-MiniLM-L6-v2` index |
| **Fallback** | Keyword scan of `.md` files in `docs_dir` (no FAISS required) |
| **Empty query** | Returns `'[DocSearch] Please provide a search query.'` |
| **No results** | Returns `'[DocSearch] No relevant documentation found.'` |
| **FAISS load fails** | Silently falls back to keyword scan (logged as WARNING) |
| **Output interpretation** | Agent uses snippets as context for the next code-writing step |


In [ ]:
from shape.tools.doc_search import DocSearchTool

# Use a non-existent index path to exercise the keyword fallback branch
ds = DocSearchTool({'doc_index_path': 'data/no_index_here',
                    'docs_dir': 'data/docs', 'doc_top_k': 3})

print(f'FAISS index loaded: {ds._vectorstore is not None}')
print(f'Keyword fallback active: {ds._vectorstore is None}')
print()

queries = [
    ('Empty query',             ''),
    ('Normal query',            'how to read a CSV file in Python'),
    ('No matching topic',       'quantum entanglement teleportation protocol'),
]
for label, q in queries:
    out = ds._run(q)
    print(f'[{label}]')
    print(f'  → {repr(out[:100])}')
    print()

## Tools 3 & 4: `linter` and `file_manager`

| Tool | Trigger | Purpose |
|---|---|---|
| `linter` | `Action: linter` | pyflakes static analysis before execution |
| `file_manager` | `Action: file_manager` | Read/write files in sandboxed workspace |

**Why lint before execute?**  
On Pi 5 (1.8 GHz ARM), one `code_executor` subprocess call takes ~380 ms even for trivial
code (Python interpreter startup). A `linter` call takes ~45 ms. Catching syntax errors and
undefined names at lint time saves the full executor cost and one ReAct iteration.


In [ ]:
from shape.tools.linter       import LinterTool
from shape.tools.file_manager import FileManagerTool
import json, time

linter = LinterTool({})
fm     = FileManagerTool({'workspace_dir': 'workspace'})

# ── Linter: catch bugs before executor ────────────────────────────────────────
print('LINTER — catches these before code_executor wastes 380 ms:')
print('-' * 60)
snippets = [
    ('Clean code',       'def add(a, b):\n    return a + b\n'),
    ('Undefined name',   'def f(x):\n    return undefined_fn(x)\n'),
    ('Classic CSV bug',  'vals.append(row["temp"])  # str not float\nsum(vals)/len(vals)\n'),
]
for label, code in snippets:
    t0  = time.perf_counter()
    out = linter._run(code)
    ms  = (time.perf_counter() - t0) * 1000
    print(f'  [{label}]  {ms:.0f} ms → {(out.strip() or "no issues")[:70]}')

# ── File manager: path traversal protection ───────────────────────────────────
print()
print(f'FILE MANAGER — workspace: {fm.workspace}')
print('-' * 60)
ops = [
    ('Write file',              {'action': 'write', 'path': 'sensor_log.txt',   'content': 'temp=22.4,aqi=312'}),
    ('Read it back',            {'action': 'read',  'path': 'sensor_log.txt'}),
    ('Path traversal attempt',  {'action': 'read',  'path': '../../etc/passwd'}),
    ('List workspace',          {'action': 'list',  'path': '.'}),
]
for label, op in ops:
    out = fm._run(json.dumps(op))
    print(f'  [{label}]: {repr(out[:70])}')

## Safety — `shape.safety.InputGuard`

InputGuard runs **before** `llm.invoke()` — it uses `re.search` on the raw user input.
If blocked, the LLM is never called. Zero Ollama compute wasted.

Three guard layers:
1. **Length**: rejects inputs > 8,000 chars (prompt injection via oversized input)
2. **Blocked patterns** (8): OS system calls, destructive shell, malware keywords, eval injection
3. **Out-of-scope** (3): medical advice, legal advice, non-coding creative writing
4. **User-defined** (`extra_blocked_patterns` in config): runtime-extensible without code changes


In [ ]:
from shape.safety import InputGuard, _BLOCKED_PATTERNS, _OUT_OF_SCOPE_PATTERNS
import time

guard = InputGuard({})

print('Pattern library:')
print(f'  Blocked patterns ({len(_BLOCKED_PATTERNS)}):')
for pat, desc in _BLOCKED_PATTERNS:
    print(f'    {pat[:55]:<55}  → {desc}')
print(f'  Out-of-scope ({len(_OUT_OF_SCOPE_PATTERNS)}):')
for pat, desc in _OUT_OF_SCOPE_PATTERNS:
    print(f'    {pat[:55]:<55}  → {desc}')

print()
cases = [
    ('Write a CSV parser',                           'ALLOW'),
    ('rm -rf /',                                     'BLOCK'),
    ('os.system("ls")',                              'BLOCK'),
    ('generate virus worm',                          'BLOCK'),
    ('Give me legal advice on my lawsuit',           'BLOCK'),
    ('How do I open a file in Python?',              'ALLOW'),
    ('A' * 8001,                                     'BLOCK'),
]

print(f'{"Input":<50} {"Expected":<10} {"Result":<45} {"Match?"}')
print('-' * 115)
for text, expected in cases:
    display = text[:48] + '..' if len(text) > 48 else text
    t0 = time.perf_counter()
    blocked, reason = guard.check(text)
    us = (time.perf_counter() - t0) * 1e6
    result  = f'BLOCKED: {reason[:35]}' if blocked else 'ALLOWED'
    correct = blocked == (expected == 'BLOCK')
    print(f'  {display:<50} {expected:<10} {result:<45} {"✓" if correct else "✗ MISMATCH"} ({us:.0f} µs)')

In [ ]:
# Alert reliability — Layer 1 is deterministic arithmetic (no LLM)
# The LLM only writes advisory text; it cannot suppress or override the alert.

from shape.sensors import parse_csv_line, validate_reading, AQI_ALERT_THRESHOLD, TEMP_ALERT_THRESHOLD

print(f'Alert thresholds: AQI > {AQI_ALERT_THRESHOLD}  |  temp > {TEMP_ALERT_THRESHOLD}°C')
print()

lines = [
    'SHAPE,14000,22.4,58.1,312,0',
    'SHAPE,16000,22.4,58.1,600,0',   # exactly at boundary — no alert
    'SHAPE,18000,22.4,58.1,601,0',   # 1 over — alert
    'SHAPE,20000,40.0,58.1,200,0',   # exactly at boundary — no alert
    'SHAPE,22000,40.1,58.1,200,0',   # 0.1 over — alert
    'SHAPE,24000,20.0,58.1,100,1',   # hardware flag alone triggers
    'GARBAGE_DATA',                   # malformed — silently dropped
]

print(f'{"CSV line":<40} {"Parsed":<8} {"Alert":<8} {"Warnings"}')
print('-' * 85)
for line in lines:
    r = parse_csv_line(line)
    if r is None:
        print(f'  {line:<40} None     —        → silently dropped')
    else:
        w = validate_reading(r) or ['none']
        print(f'  {line:<40} OK       {str(r.is_alert):<8} {w[0]}')

In [ ]:
# LIVE FIELDWORK TASK — exercises all 4 tools in one agent call
# Generate sensor data, write to workspace, ask agent to process it.

import csv, random, json
from pathlib import Path
from datetime import datetime, timedelta
from shape.agent import build_agent
import logging, os
logging.basicConfig(level=logging.WARNING)

random.seed(42)
rows = []
start = datetime(2026, 11, 1, 6, 0, 0)
for i in range(20):
    aqi_base = 320 + (i - 11) * 25 if i >= 11 else 320
    rows.append({
        'timestamp': (start + timedelta(minutes=i*15)).isoformat(),
        'temp_c': round(random.gauss(22, 3), 2),
        'humidity_pct': round(random.gauss(55, 8), 1),
        'co2_ppm': max(280, round(random.gauss(aqi_base, 20))),
    })

Path('workspace').mkdir(exist_ok=True)
with open('workspace/sensor_data.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['timestamp','temp_c','humidity_pct','co2_ppm'])
    w.writeheader(); w.writerows(rows)

print(f'sensor_data.csv written: {len(rows)} rows')

AGENT_CONFIG = {
    'model': 'codellama', 'ollama_url': 'http://localhost:11434',
    'temperature': 0.2, 'context_window': 4096, 'memory_window': 6,
    'max_iterations': 8, 'executor_timeout': 10,
    'feedback_path': 'data/feedback.jsonl', 'workspace_dir': 'workspace',
    'doc_index_path': 'data/doc_index', 'docs_dir': 'data/docs', 'doc_top_k': 3,
    'verbose': True,
}

agent = build_agent(AGENT_CONFIG)

task = '''
I have workspace/sensor_data.csv with columns: timestamp, temp_c, humidity_pct, co2_ppm.
Write Python code that:
1. Reads the CSV using the file_manager tool
2. Computes min, max, and mean for each numeric column
3. Flags any row where co2_ppm > 430 as ALERT
4. Prints a formatted summary
Lint first, then run it.
'''

print('=' * 65)
print('FIELDWORK TASK — file_manager + linter + code_executor')
print('=' * 65)
response = agent.run(task)
print('\n' + '=' * 65)
print('RESPONSE:')
print('=' * 65)
print(response)

## Summary

| Item | Demonstrated |
|---|---|
| Tool 1: `code_executor` | All error paths: normal, empty, timeout, syntax err, runtime err, cap |
| Tool 2: `doc_search` | FAISS primary path + keyword fallback + all error returns |
| Tool 3: `linter` | Timing comparison (45 ms vs 380 ms), 3 bug categories caught |
| Tool 4: `file_manager` | Write/read cycle + path traversal blocked |
| Safety (pre-LLM) | 7 test cases, timing shown (~µs), all branches validated |
| Sensor CSV parsing | `parse_csv_line` range checks, malformed input dropped |
| Live fieldwork task | All 4 tools exercised in one agent call |

▶ **Continue to `04_SHAPE_Evaluation_Demo.ipynb`**
